# arcedge S2-M1 acceptance benchmark

Produces the Stage 2 milestone S2-M1 metrics in one run:

1. **Full-solve equivalence** on the SF 1.68M-arc time-expanded instance
   (K=150): `dijkstra` / `dag` / `cuda` must agree (CPU backends to machine
   precision, CUDA within FP32 tolerance + FP64-certified final bound).
2. **Batch-scaling sweep**: pure SSSP throughput (`--no-primal`) for
   K = 100 ... 2000 commodities. The GPU relaxes the whole batch per kernel
   launch; CPU backends scale only with cores.

The last cell prints a paste-ready results table and writes `bench_s2m1.json`
— send either back to record in `PLAN.md`.

**Runtime → Change runtime type → GPU** (T4 fine, A100 better).
Total runtime ≈ 10–15 min, dominated by the CPU side of the sweep.

In [ ]:
!nvidia-smi -L
!nvidia-smi | head -12

In [ ]:
import os
BRANCH = 'claude/stage-1-implementation-plan-9lv5f3'  # or 'main' once merged
if os.path.exists('/content/arcedge'):
    %cd /content/arcedge
    !git pull
else:
    token = ''  # <-- paste a GitHub token here if the repo is private
    url = f'https://{token}@github.com/fhk/arcedge.git' if token else 'https://github.com/fhk/arcedge.git'
    !git clone --branch {BRANCH} {url} /content/arcedge
    %cd /content/arcedge
!git log --oneline -1

In [ ]:
!pip install -q highspy numpy pyarrow shapely
!cmake -B build -DCMAKE_BUILD_TYPE=Release -DARCEDGE_CUDA=ON 2>&1 | tail -2
!cmake --build build -j2 2>&1 | tail -2
!ctest --test-dir build --output-on-failure | tail -4

## Metric 1 — full-solve equivalence and end-to-end time (K = 150)

All three backends solve the same instance to the same certified bounds.
The `cuda` run is shown unmuted so the `fp64 certification:` line is visible
(the FP32 GPU bound re-proven in FP64 on CPU at the best multipliers).

In [ ]:
import subprocess, re, json

def run_solve(instance, backend, quiet=True, extra=()):
    cmd = ['./build/arcedge', 'solve', instance, '--iters', '120', '--tol', '0.01',
           '--primal-every', '10', '--sp-backend', backend] + \
          (['--quiet'] if quiet else []) + list(extra)
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    line = [l for l in out.splitlines() if l.startswith('result:')][-1]
    m = re.search(r'lb ([\d.]+)\s+ub ([\d.]+)\s+gap ([\d.]+)%\s+iters (\d+)\s+time ([\d.]+) ms', line)
    cert = next((l for l in out.splitlines() if l.startswith('fp64')), None)
    return dict(lb=float(m[1]), ub=float(m[2]), gap=float(m[3]),
                iters=int(m[4]), ms=float(m[5]), cert=cert)

!mkdir -p data
!./build/arcedge gen --street data/sf_streets.graph --out data/sf_te.txt \
    --time 36 --commodities 150 --cap 8 --hubs 5 --hub-frac 0.5 --seed 17

full = {}
for b in ['dijkstra', 'dag', 'cuda']:
    full[b] = run_solve('data/sf_te.txt', b, quiet=(b != 'cuda'))
    print(f"{b:9s} lb {full[b]['lb']:.2f}  ub {full[b]['ub']:.2f}  "
          f"gap {full[b]['gap']:.4f}%  iters {full[b]['iters']}  {full[b]['ms']:.0f} ms")

ref = full['dijkstra']
assert abs(full['dag']['lb'] - ref['lb']) <= 1e-9 * ref['lb'], 'dag LB mismatch'
assert abs(full['cuda']['lb'] - ref['lb']) <= 1e-4 * ref['lb'], 'cuda LB beyond FP32 tolerance'
assert abs(full['cuda']['ub'] - ref['ub']) <= 1e-4 * ref['ub'], 'cuda UB beyond FP32 tolerance'
print('\nEQUIVALENCE OK')
print(full['cuda']['cert'])

## Metric 2 — batch-scaling sweep (pure SSSP throughput)

`--no-primal` disables the sequential CPU heuristic and the certification
pass, so wall-clock ≈ iterations × batched shortest paths. `dijkstra` is
skipped above K = 500 (per-commodity heap search gets slow on 2 vCPUs and
adds nothing past that point). GPU memory peaks at K = 2000:
2000 × 413,532 nodes × 8 B ≈ 6.6 GB.

In [ ]:
KS = [100, 250, 500, 1000, 2000]
ITERS = 10
sweep = {'dijkstra': {}, 'dag': {}, 'cuda': {}}
for K in KS:
    inst = f'data/sf_te_k{K}.txt'
    if not os.path.exists(inst):
        !./build/arcedge gen --street data/sf_streets.graph --out {inst} \
            --time 36 --commodities {K} --cap 50 --hubs 8 --hub-frac 0.5 --seed 17
    backends = ['dijkstra', 'dag', 'cuda'] if K <= 500 else ['dag', 'cuda']
    for b in backends:
        out = subprocess.run(
            ['./build/arcedge', 'solve', inst, '--iters', str(ITERS),
             '--no-primal', '--sp-backend', b, '--quiet'],
            capture_output=True, text=True).stdout
        ms = float(re.search(r'time ([\d.]+) ms', out)[1])
        sweep[b][K] = ms / ITERS
        print(f'K={K:5d}  {b:9s} {ms / ITERS:9.1f} ms/iter')

In [ ]:
import matplotlib.pyplot as plt

print(f"{'K':>6} {'dijkstra':>11} {'dag':>11} {'cuda':>11} {'cuda vs dag':>12}")
for K in KS:
    dj = f"{sweep['dijkstra'][K]:11.1f}" if K in sweep['dijkstra'] else ' ' * 10 + '-'
    print(f"{K:6d} {dj} {sweep['dag'][K]:11.1f} {sweep['cuda'][K]:11.1f} "
          f"{sweep['dag'][K] / sweep['cuda'][K]:11.2f}x")

for b, style in [('dijkstra', 'o-'), ('dag', 's-'), ('cuda', '^-')]:
    ks = sorted(sweep[b])
    if ks:
        plt.loglog(ks, [sweep[b][k] for k in ks], style, label=b)
plt.xlabel('batch size K (commodities)')
plt.ylabel('ms per subgradient iteration')
plt.title('Batched SSSP: SF time-expanded graph, 1.68M arcs')
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.show()

In [ ]:
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
results = dict(gpu=gpu, full_solve=full, sweep_ms_per_iter=sweep, sweep_iters=ITERS)
json.dump(results, open('bench_s2m1.json', 'w'), indent=2)
print('saved bench_s2m1.json')
print()
print('--- paste-ready S2-M1 results ---')
print(f'measured on: {gpu}')
print()
print('full solve K=150:', {b: f"{v['gap']:.3f}% gap, {v['iters']} iters, {v['ms']:.0f} ms"
                            for b, v in full.items()})
print()
print('| K | dag ms/iter (CPU) | cuda ms/iter | speedup |')
print('|---:|---:|---:|---:|')
for K in KS:
    print(f"| {K} | {sweep['dag'][K]:.0f} | {sweep['cuda'][K]:.0f} "
          f"| {sweep['dag'][K] / sweep['cuda'][K]:.2f}x |")

## Done

Send back `bench_s2m1.json` (left sidebar → Files → download) or just the
printed *paste-ready* block above. Acceptance criteria for S2-M1:

- equivalence assertions pass (they already failed loudly above if not),
- the FP64 certification line shows a valid certified bound, and
- cuda ms/iter beats dag ms/iter with a margin that **grows with K**.